In [1]:
# Step 1: LLM
from langchain.chat_models import init_chat_model

llm_model = init_chat_model(model="gpt-4o")

In [2]:
# Step 2: Prompt — no {context}, just history + question
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, friendly assistant. Answer naturally, using "
               "the conversation history to maintain context."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

In [3]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [4]:
# Step 3: Simple chain
base_chain = prompt_template | llm_model | parser

In [5]:
# Step 4: In-memory store for chat histories, keyed by session_id
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [6]:
# Step 5: Wrap with memory
from langchain_core.runnables.history import RunnableWithMessageHistory

# for RAG use
# base_chain = (
#     RunnableParallel(
#         context=lambda x: format_docs(retriever.invoke(x["question"])),
#         question=lambda x: x["question"],
#         chat_history=lambda x: x.get("chat_history", []),
#     )
#     | prompt_template
#     | llm_model
#     | StrOutputParser()
# )

chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

c:\Users\Dell\Desktop\RAG Course\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
config = {"configurable": {"session_id": "user1"}}

In [8]:
response1 = chain_with_memory.invoke({"question": "My name is Acchint."}, config=config)
print(response1)

response2 = chain_with_memory.invoke({"question": "What's my name?"}, config=config)
print(response2)

C:\Users\Dell\AppData\Local\Temp\ipykernel_18504\1713670687.py:8: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


Nice to meet you, Acchint! How can I assist you today?
Your name is Acchint!


### Multiple sessions

In [9]:
config2 = {"configurable": {"session_id": "user2"}}

In [10]:
response_user2 = chain_with_memory.invoke({"question": "What's my name?"}, config=config2)
response_user2

"I'm sorry, but I don't know your name from our conversation history. If you'd like to tell me, I'd be happy to use it!"

### Inspect stored history

In [11]:
history = get_session_history("user1")
for msg in history.messages:
    print(f"{msg.type}: {msg.content}")

human: My name is Acchint.
ai: Nice to meet you, Acchint! How can I assist you today?
human: What's my name?
ai: Your name is Acchint!


### Persistent storage across kernel restarts

In [ ]:
# from langchain_community.chat_message_histories import FileChatMessageHistory

# def get_session_history(session_id: str):
#     return FileChatMessageHistory(f"./chat_histories/{session_id}.json")

C:\Users\Dell\AppData\Local\Temp\ipykernel_18504\1088419572.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import FileChatMessageHistory


In [ ]:
# history = get_session_history("user1")
# for msg in history.messages:
#     print(f"{msg.type}: {msg.content}")